### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

In [1]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### Defaults

In [2]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


['programs.txt']

### Get GDC programs - get_gdc_progams()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "facets": "program.name"  

### Get valid primary sites - get_primary_sites()

> end_point: url_gdc_project = "https://api.gdc.cancer.gov/projects"  
> "fields": "project_id,name,primary_site,disease_type"  

### Get valid subtypes - get_subtypes()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> facets = "diagnoses.primary_diagnosis"  

### For each subtype → get stages  - get_stages()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> "field": "diagnoses.ajcc_pathologic_stage" - AJCC diagnoses   

### For each (subtype, stage) → get_samples()

> end_point: url_gdc_cases = "https://api.gdc.cancer.gov/cases"  
> given pid, subtype, stage  
> from cases   
> "samples.sample_id", "samples.submitter_id", "samples.sample_type"  

### get RNA-seq files - get_expression_files_given_samples()

> given: "field": "cases.case_id", "value": case_ids  
> end_point: url_gdc_files = "https://api.gdc.cancer.gov/files"  



### All programs

In [3]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [4]:
prog_list

['TCGA',
 'MATCH',
 'TARGET',
 'CGCI',
 'CMI',
 'APOLLO',
 'BEATAML1.0',
 'CPTAC',
 'MP2PRT',
 'ALCHEMIST',
 'CCDI',
 'CCG',
 'CDDP_EAGLE',
 'CTSP',
 'EXCEPTIONAL_RESPONDERS',
 'FM',
 'HCMI',
 'MMRF',
 'NCICCR',
 'OHSU',
 'ORGANOID',
 'RC',
 'REBC',
 'TRIO',
 'VAREPOP',
 'WCDT']

In [5]:
force=False
verbose=True

program = 'TCGA'

dfc = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

dfc.head(3)


No data found for 'TCGA'. error: 'data'


""


In [10]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [12]:

filters = {
            "op": "in",
            "content": { "field": "program.name", "value": [program] }
        }

params = {
    "filters": json.dumps(filters),
    "fields": "project_id,name,primary_site,disease_type",
    "format": "JSON",
    "size": 1000
}

res = requests.get(gdc.url_gdc_project, params=params)
res_json = res.json()

if "data" in res_json:
    data = res_json["data"]["hits"]
    print("OK:", data)
else:
    print("Error:", res_json)  # debug

OK: [{'id': 'TCGA-LGG', 'primary_site': ['Brain'], 'project_id': 'TCGA-LGG', 'disease_type': ['Gliomas'], 'name': 'Brain Lower Grade Glioma'}, {'id': 'TCGA-BRCA', 'primary_site': ['Breast'], 'project_id': 'TCGA-BRCA', 'disease_type': ['Adnexal and Skin Appendage Neoplasms', 'Adenomas and Adenocarcinomas', 'Ductal and Lobular Neoplasms', 'Cystic, Mucinous and Serous Neoplasms', 'Complex Epithelial Neoplasms', 'Fibroepithelial Neoplasms', 'Squamous Cell Neoplasms', 'Epithelial Neoplasms, NOS', 'Basal Cell Neoplasms'], 'name': 'Breast Invasive Carcinoma'}, {'id': 'TCGA-LAML', 'primary_site': ['Hematopoietic and reticuloendothelial systems'], 'project_id': 'TCGA-LAML', 'disease_type': ['Myeloid Leukemias'], 'name': 'Acute Myeloid Leukemia'}, {'id': 'TCGA-UCS', 'primary_site': ['Uterus, NOS', 'Corpus uteri'], 'project_id': 'TCGA-UCS', 'disease_type': ['Basal Cell Neoplasms', 'Complex Mixed and Stromal Neoplasms'], 'name': 'Uterine Carcinosarcoma'}, {'id': 'TCGA-GBM', 'primary_site': ['Brain

In [ ]:
df_ps = pd.DataFrame(data)
cols = ["project_id", "primary_site", "disease_type"]
df_ps = df_ps[cols]

df_ps = df_ps.sort_values(["primary_site", "disease_type"])

### Subtypes

In [ ]:
force=False
verbose=True

i=1
pid = dfc.iloc[i].project_id
primary_site = dfc.iloc[i].primary_site

print(pid, primary_site)

df_subt = gdc.get_subtypes(pid=pid, do_filter=True, force=force, verbose=verbose)

print(len(df_subt))
df_subt

### Stages

In [ ]:
force=False
verbose=True

i=0
subtype = df_subt.iloc[i].subtype_raw

print(pid, subtype)

df_stage = gdc.get_stages(pid=pid, subtype=subtype, do_filter=True, force=force, verbose=verbose)

print(len(df_stage))
df_stage

In [ ]:
force=False
verbose=True

i=0
stage = df_stage.iloc[i].stage_raw


print(pid, subtype, stage)

df_case = gdc.get_cases(pid=pid, subtype=subtype, stage=stage, force=force, verbose=verbose)
print(len(df_case))
df_case.head(12)

### Given a PS, Subtype, and Stage --> return the SAMPLES

In [ ]:
force=False
verbose=True

print(pid, subtype, stage)

df_sample = gdc.get_samples(pid=pid, subtype=subtype, stage=stage, batch_size=200, force=force, verbose=verbose)
print(len(df_sample))
df_sample.head(8)

In [ ]:
force=False
verbose=True

case_ids = list(np.unique(df_sample.case_id))

print(pid, subtype, stage, f"cases {len(case_ids)}")

df_files = gdc.get_expression_files_given_samples(pid=pid, subtype=subtype, stage=stage, 
                                                  case_ids=case_ids, batch_size=20, force=force, verbose=verbose)
print(len(df_files))
df_files.head(6)

In [ ]:
case_ids[:10]